In [ ]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import from_json

In [9]:
spark = SparkSession.builder.master('local[*]').appName('12').getOrCreate()

- **json** 장점
> - 행마다 데이터 항목이 달라도 수용 가능( ex> height만 있는 행, height와 weight가 모두 있는 행을 수용 가능)
> - 객체 안에 객체 또는 리스트를 포함할 수 있으므로, 복잡한 관계를 직관적으로 표현함
> - Key-Value 쌍 구조이므로 컴퓨터가 parsing하기 효율적
> - python, java, C등 모든 현대언어에서 json 지원

In [10]:
str_data = [
    ('Tom','{"height":173.5,"weight":69.8}'),
    ('Jane','{"height":163.0,"weight":45.5,"blood_p":110.0}')
]

In [11]:
data_schema = ['name','properties']

In [12]:
df = spark.createDataFrame(str_data,data_schema)

In [13]:
df.show(truncate=False)
df.printSchema()

+----+----------------------------------------------+
|name|properties                                    |
+----+----------------------------------------------+
|Tom |{"height":173.5,"weight":69.8}                |
|Jane|{"height":163.0,"weight":45.5,"blood_p":110.0}|
+----+----------------------------------------------+

root
 |-- name: string (nullable = true)
 |-- properties: string (nullable = true)



- `from_json`은 **표준 json 문자열 형태**로 표현된 문자열로 부터, `MapType` 또는 형식에 맞는 `StructType`으로 변환할 수 있음
- **표준 json 문자열**: 중괄호(`{}`), 콜론(`:`), Key는 쌍따옴표(`""`) 사용

### (1) **MapType**
- `from_json(Col, type)`: Col이 json 문자열 형태의 값을 저장하고 있는 경우, 지정된 타입(일반적으로 `MapType` 또는 `StructType`)으로 로드함

In [17]:
map_df = df.withColumn('profile',from_json(df.properties,MapType(StringType(),FloatType())))

In [18]:
map_df.show(truncate=False)
map_df.printSchema()

+----+----------------------------------------------+---------------------------------------------------+
|name|properties                                    |profile                                            |
+----+----------------------------------------------+---------------------------------------------------+
|Tom |{"height":173.5,"weight":69.8}                |{height -> 173.5, weight -> 69.8}                  |
|Jane|{"height":163.0,"weight":45.5,"blood_p":110.0}|{height -> 163.0, weight -> 45.5, blood_p -> 110.0}|
+----+----------------------------------------------+---------------------------------------------------+

root
 |-- name: string (nullable = true)
 |-- properties: string (nullable = true)
 |-- profile: map (nullable = true)
 |    |-- key: string
 |    |-- value: float (valueContainsNull = true)



### (2) **StructType**
- height, weight, blood_p 각각의 sub Column을 지정하는 형태 

In [19]:
struct_schema = StructType([
    StructField('height',FloatType()),
    StructField('weight',FloatType()),
    StructField('blood_p',DoubleType())
])

In [20]:
struct_df = df.withColumn('profile',from_json(df.properties,struct_schema))

In [21]:
struct_df.show(truncate=False)
struct_df.printSchema()

+----+----------------------------------------------+--------------------+
|name|properties                                    |profile             |
+----+----------------------------------------------+--------------------+
|Tom |{"height":173.5,"weight":69.8}                |{173.5, 69.8, NULL} |
|Jane|{"height":163.0,"weight":45.5,"blood_p":110.0}|{163.0, 45.5, 110.0}|
+----+----------------------------------------------+--------------------+

root
 |-- name: string (nullable = true)
 |-- properties: string (nullable = true)
 |-- profile: struct (nullable = true)
 |    |-- height: float (nullable = true)
 |    |-- weight: float (nullable = true)
 |    |-- blood_p: double (nullable = true)

